In [ ]:
from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }a
</style>
"""))


In [ ]:
import matplotlib.pyplot as plt 

import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
# Get data of labor
sex_work = pd.read_stata( r'../_data/8_trab_sex_20.dta' )
sex_work[ 'dpt_code' ] = sex_work.cod_ubigeo.str[ :2 ].copy()
sex_work[ 'prov_code' ] = sex_work.cod_ubigeo.str[ :4 ].copy()

# Sex work
women_work = sex_work[ sex_work.sex == 'Mujer' ].copy().reset_index( drop = True )

In [ ]:
# get data from lima
women_work[ 'month' ] = pd.to_datetime( women_work.month , format = '%B' ) \
                                    .dt.strftime( '%m' ) \
                                    .astype( int )

In [ ]:
# Sort by department and month
women_work.sort_values([ 'dpt_code', 'month'], inplace = True )

# Get the total number of women workers by dpt
dpt_women_work = women_work.groupby( [ 'dpt_code', 'month'], as_index = False )[['empl']] \
                            .sum() \
                            .rename( columns = {'empl' :'women_empl'})

# Sort by dpt code and month
dpt_women_work.sort_values([ 'dpt_code', 'month'], inplace = True )

In [ ]:
df2 = dpt_women_work.groupby( ['dpt_code'],as_index = False )[['women_empl']].mean()

## Shapefile

In [ ]:
dpt_shp = gpd.read_file( r'../_data/INEI_LIMITE_DEPARTAMENTAL/INEI_LIMITE_DEPARTAMENTAL.shp' )

In [ ]:
df3 = dpt_shp.merge( df2, left_on = 'CCDD', right_on = 'dpt_code'  )

In [ ]:
df3.plot( column='women_empl', cmap='Reds', figsize=(20, 20), linestyle='--',
                      edgecolor='black', 
                      legend = True)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))
df3.plot( ax = ax, 
        column='women_empl', 
         cmap= 'Reds', 
         figsize=(20, 20), 
         linestyle='--',
         edgecolor='black', 
         legend = True,  
         scheme = "User_Defined", 
         classification_kwds = dict( bins = [ 20000, 40000, 60000, 100000  ] ), 
         legend_kwds = dict(  loc='upper left',
                            bbox_to_anchor=(1.01, 1),
                            fontsize='x-large',
                            title= "Number of Employers", 
                            title_fontsize = 'x-large', 
                            frameon= False )
        )


In [ ]:
df3[ df3.CCDD != "15" ].plot( column='women_empl', cmap='Reds', figsize=(20, 20), linestyle='--',
                      edgecolor='black', 
                      legend = True)

In [ ]:
df4 = sex_work.groupby( ['dpt_code', 'month', 'sex'], as_index = False )[['empl']].sum() \
        .pivot( index = [ 'dpt_code', 'month' ] , columns = 'sex',values='empl') \
        .reset_index()

In [ ]:
df4

In [ ]:
df4[ 'prop_wom' ] = ( df4.Mujer * 100 / df4.Hombre )

In [ ]:
df5 = dpt_shp.merge( df4, left_on = 'CCDD', right_on = 'dpt_code'  )

In [ ]:
fig, axis = plt.subplots( nrows = 4, ncols= 3, figsize = ( 15, 15 ) )

idx = 0
for i in range( 4 ):
    for j in range ( 3 ):
        
        
        ax = axis[ i ][ j ]
        
        month = df5.month.unique()[ idx ]
        
        df6 = df5[ df5.month == month ]
        
        df6.plot( column='prop_wom', 
                  cmap='Reds', 
                  linestyle='--',
                  edgecolor='black', 
                  legend = True, 
                  ax = ax 
                )
        
        ax.set_title( month )
        
        idx = idx + 1

In [ ]:
from textwrap import wrap

In [ ]:
# Inverting colour map
cmap = plt.cm.OrRd

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))
df6.plot( ax = ax, 
        column='prop_wom', 
         cmap= cmap, 
         figsize=(20, 20), 
         linestyle='--',
         edgecolor='black', 
         legend = True,  
         scheme = "User_Defined", 
         classification_kwds = dict( bins = [ 20, 30, 40, 50,  100 ] ), 
         legend_kwds=dict(  loc='upper left',
                            bbox_to_anchor=(1.01, 1),
                            fontsize='x-large',
                            title= "Women Proportion", 
                            title_fontsize = 'x-large', 
                            frameon= False )
        )


In [ ]:
df6.loc[ (df6.NOMBDEP == 'LIMA'), 'prop_wom' ] = np.nan

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))
df6.plot( ax = ax, 
        column='prop_wom', 
         cmap= cmap, 
         figsize=(20, 20), 
         linestyle='--',
         edgecolor='black', 
         legend = True,  
         scheme = "User_Defined", 
         missing_kwds= dict(color = "#DADADB",), 
         classification_kwds = dict( bins = [ 20, 30, 40, 50,  100 ] ), 
         legend_kwds=dict(  loc='upper left',
                            bbox_to_anchor=(1.01, 1),
                            fontsize='x-large',
                            title= "Women Proportion", 
                            title_fontsize = 'x-large', 
                            frameon= False )
        )

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))
dpt_shp.plot( ax = ax )

In [ ]:
dpt_shp['country'] = 'PERU'

In [ ]:
ctr_shp = dpt_shp.dissolve( by = 'country')

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))
ctr_shp.plot( ax = ax )

Generating grids

In [ ]:
# Import a Shapefile
dist_shp = gpd.read_file(r'../_data/shape_file/DISTRITOS.shp')
dist_shp["geometry"][0]

In [ ]:
# Keep only smp geometry
smp_geo = dist_shp.query( "`PROVINCIA` == 'LIMA' & `DISTRITO` == 'SAN MARTIN DE PORRES'").geometry

In [ ]:
# Generating grids
xmin, ymin, xmax, ymax= smp_geo.total_bounds

In [ ]:
smp_geo.crs

In [ ]:
import shapely.geometry


In [ ]:
# how many cells across and down
n_cells = 80
cell_size = (xmax-xmin)/n_cells
# projection of the grid
crs = 4326

# create the cells in a loop
grid_cells = []
for x0 in np.arange(xmin, xmax+cell_size, cell_size ):
    for y0 in np.arange(ymin, ymax+cell_size, cell_size):
        # bounds
        x1 = x0-cell_size
        y1 = y0+cell_size
        grid_cells.append( shapely.geometry.box(x0, y0, x1, y1)  )
cell = gpd.GeoDataFrame(grid_cells, columns=['geometry'], 
                                 crs=crs)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
smp_geo.plot()

In [ ]:
ax = smp_geo.plot(markersize=.1, figsize=(12, 8), )
plt.autoscale(False)
cell.plot( ax = ax , facecolor="none", edgecolor='grey')
ax.axis("off")


### Get Information from Raster

In [ ]:

from rasterstats import zonal_stats
import geopandas as gpd

# Load your vector data
dpt_shp = gpd.read_file( r'../_data/INEI_LIMITE_DEPARTAMENTAL/INEI_LIMITE_DEPARTAMENTAL.shp' )

# Specify your raster file
raster_path = '../_data/VIIRS_NTL_Peru_YearlyComposite_2021.tif'  # Change this to your raster file path

# Calculate zonal statistics
stats = zonal_stats(dpt_shp, raster_path, stats=["count", "min", "mean", "max", "sum"])

# The 'stats' variable is a list of dictionaries with the statistics for each feature in the vector file
# For example, to print the statistics for the first feature:
print(stats[0])


In [ ]:
stats_gdf = pd.DataFrame(stats)
df1 = pd.concat([dpt_shp, stats_gdf], axis = 1)